In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/df_final_with_genres.csv")

df[df["genres"].isna()].shape

(160910, 7)

In [5]:
df[df["genres"].isna()]["book_id"].nunique()

585

In [6]:
import pandas as pd
import ast

print("\nDataset shape:")
print(df.shape)


# -------------------------------------------------
# Genre parser (same logic as your SVD script)
# -------------------------------------------------
def parse_genres(s):

    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip().strip('"').strip("'") for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip().strip('"').strip("'") for t in s.split(sep) if t.strip()]

    return [s.strip().strip('"').strip("'")]


# -------------------------------------------------
# Collect statistics
# -------------------------------------------------

all_genres = set()
genre_book_map = {}
books_with_no_genre = set()

for _, row in df.iterrows():

    book = row["book_id"]
    parsed = parse_genres(row["genres"])

    if len(parsed) == 0:
        books_with_no_genre.add(book)
        continue

    for g in parsed:

        all_genres.add(g)

        if g not in genre_book_map:
            genre_book_map[g] = set()

        genre_book_map[g].add(book)


# -------------------------------------------------
# Unique genres
# -------------------------------------------------

print("\nUnique genres:")
for g in sorted(all_genres):
    print(g)

print("\nNumber of unique genres:", len(all_genres))


# -------------------------------------------------
# Unique books per genre
# -------------------------------------------------

print("\nUnique books per genre:")

genre_counts = []

for g in sorted(genre_book_map.keys()):
    count = len(genre_book_map[g])
    genre_counts.append((g, count))

genre_df = pd.DataFrame(genre_counts, columns=["genre", "unique_books"])
genre_df = genre_df.sort_values("unique_books", ascending=False)

print(genre_df)


# -------------------------------------------------
# Missing genres
# -------------------------------------------------

print("\nBooks with missing / unparsable genres:")
print(len(books_with_no_genre))


# -------------------------------------------------
# Dataset summary
# -------------------------------------------------

print("\nTotal unique books in dataset:")
print(df["book_id"].nunique())


Dataset shape:
(5976479, 7)

Unique genres:
Adult
Adventure
Children's
Classics
Drama
Fantasy
Historical
Horror
Mystery
Nonfiction
Romance
Science Fiction
Thriller

Number of unique genres: 13

Unique books per genre:
              genre  unique_books
4             Drama          3006
8           Mystery          2563
10          Romance          2131
5           Fantasy          2088
1         Adventure          1789
12         Thriller          1606
9        Nonfiction          1071
3          Classics           901
2        Children's           863
6        Historical           857
11  Science Fiction           855
7            Horror           769
0             Adult           331

Books with missing / unparsable genres:
585

Total unique books in dataset:
10000


In [8]:
import pandas as pd
import ast



def parse_genres(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# rows where genre parsing fails
invalid_rows = df[df["genres"].apply(lambda x: len(parse_genres(x)) == 0)]

print("Number of rows with invalid genres:", len(invalid_rows))

print("\nExample rows (full rows):\n")
print(invalid_rows.head(10))

Number of rows with invalid genres: 160910

Example rows (full rows):

      user_id  book_id  rating         decade original_title authors genres
1           2     4081       4           2000            NaN     NaN    NaN
433        32     1383       3           1970            NaN     NaN    NaN
438        32       75       3           1990            NaN     NaN    NaN
584        31       75       3           1990            NaN     NaN    NaN
1003       76       75       5           1990            NaN     NaN    NaN
1034       25     8555       4           1960            NaN     NaN    NaN
1297       78     4106       4  Ancient Books            NaN     NaN    NaN
1561       24     2757       3           1990            NaN     NaN    NaN
1619      113     3244       5           2000            NaN     NaN    NaN
1637      113     5629       4           1990            NaN     NaN    NaN


## clean dataset

In [9]:
import pandas as pd
import ast

INPUT_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/df_final_with_genres.csv"
OUTPUT_PATH = "/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv"

df = pd.read_csv(INPUT_PATH)

print("Original dataset shape:", df.shape)


# -------------------------------------------------
# Genre parser (same logic as your SVD code)
# -------------------------------------------------
def parse_genres(s):

    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# -------------------------------------------------
# Identify books with no valid genre
# -------------------------------------------------

df["parsed_genres"] = df["genres"].apply(parse_genres)

invalid_books = df[df["parsed_genres"].apply(len) == 0]["book_id"].unique()

print("Books with no valid genre:", len(invalid_books))


# -------------------------------------------------
# Remove those books from the dataset
# -------------------------------------------------

df_clean = df[~df["book_id"].isin(invalid_books)].copy()

# drop helper column
df_clean.drop(columns=["parsed_genres"], inplace=True)

print("Clean dataset shape:", df_clean.shape)


# -------------------------------------------------
# Save cleaned dataset
# -------------------------------------------------

df_clean.to_csv(OUTPUT_PATH, index=False)

print("\nClean dataset saved as:", OUTPUT_PATH)

Original dataset shape: (5976479, 7)
Books with no valid genre: 585
Clean dataset shape: (5815569, 7)

Clean dataset saved as: /home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv


In [10]:
import pandas as pd
import ast

df = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/SVD/data/genre_clean.csv")

def parse_genres(s):
    if pd.isna(s):
        return []

    s = str(s).strip()

    if not s:
        return []

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except:
            return []

    for sep in [",", "|", ";", "//", "/"]:
        if sep in s:
            return [t.strip() for t in s.split(sep) if t.strip()]

    return [s]


# rows where genre parsing fails
invalid_rows = df[df["genres"].apply(lambda x: len(parse_genres(x)) == 0)]

print("Number of rows with invalid genres:", len(invalid_rows))

print("\nExample rows (full rows):\n")
print(invalid_rows.head(10))

Number of rows with invalid genres: 0

Example rows (full rows):

Empty DataFrame
Columns: [user_id, book_id, rating, decade, original_title, authors, genres]
Index: []
